# Nykaa Annual Report Chatbot

In [2]:
!pip install -q langchain==0.1.16 langchain-community==0.0.33 langchain-groq \
    langchain-text-splitters chromadb sentence-transformers python-dotenv pypdf


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GROQ_API_KEY"), "Missing GROQ_API_KEY - add it to a .env file in this folder"
print("Groq API key loaded ✅")

Groq API key loaded ✅


## 3. Load and split the PDF

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = "Nykaa_Integrated-Annual-Report-2024-25.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

# Smaller chunks + smaller overlap = less redundant context fed to the LLM,
# which is one of the reasons answers were bloated before.
splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
docs = splitter.split_documents(pages)

print(f"Loaded {len(pages)} pages -> split into {len(docs)} chunks")

Loaded 331 pages -> split into 2065 chunks


## 4. Build the vector store (embeddings)


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma.from_documents(
    docs,
    embedding_model,
    persist_directory="models/chroma_db"
)

print("Vector DB ready ✅")

C:\Users\payal\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector DB ready ✅


## 5. Build the retriever + LLM + QA chain


In [6]:
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq

# Fewer, more targeted chunks retrieved -> less noisy context -> shorter, more focused answers
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 12}
)

prompt_template = """You are a precise financial analyst answering questions about Nykaa's annual report.

Use ONLY the context below. Answer in 3-5 short bullet points, each one line, no sub-bullets.
State any numbers/percentages/names first, plainly, with no preamble like \"Based on the context\".
Do not repeat a point. Do not add generic commentary or disclaimers.
If the answer isn't in the context, reply exactly: \"This information is not available in the report.\"

<CONTEXT>
{context}
</CONTEXT>

Question: {question}

Answer (max 5 bullet points, max ~80 words total):"""

prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # free on Groq, no credit card
    temperature=0,
    max_tokens=250,   # hard cap so the model physically cannot ramble
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)

print("✅ RAG pipeline ready")

✅ RAG pipeline ready


## 6. Ask questions


In [15]:
def ask(question: str):
    result = qa_chain.invoke({"query": question})
    answer = result["result"].strip()
    sources = sorted({doc.metadata.get("page", "?") for doc in result["source_documents"]})
    print(f"Q: {question}\n")
    print(f"A:\n{answer}\n")
    print(f"📄 Source pages: {sources}")
    print("-" * 60)
    return answer

In [16]:
ask("What are the key financial highlights of Nykaa in 2024-25?")

Q: What are the key financial highlights of Nykaa in 2024-25?

A:
* $935 million revenue 
* 24% growth 
* 6% EBITDA margin 
* 11.3% ROCE 
* 1.6% Capex as percentage of revenue

📄 Source pages: [8, 10, 11, 226]
------------------------------------------------------------


'* $935 million revenue \n* 24% growth \n* 6% EBITDA margin \n* 11.3% ROCE \n* 1.6% Capex as percentage of revenue'

In [17]:
ask("What are the main growth drivers for Nykaa?")

Q: What are the main growth drivers for Nykaa?

A:
24% growth in consolidated revenue 
37% increase in EBITDA 
6% EBITDA margin 
54 bps YoY increase in EBITDA margin 
H7,950 crores consolidated revenue

📄 Source pages: [8, 48, 127, 146]
------------------------------------------------------------


'24% growth in consolidated revenue \n37% increase in EBITDA \n6% EBITDA margin \n54 bps YoY increase in EBITDA margin \nH7,950 crores consolidated revenue'

In [26]:
ask("What is the EBITDA margin of Nykaa in FY2024-FY2025?")

Q: What is the EBITDA margin of Nykaa in FY2024-FY2025?

A:
* 5.4% in FY2024
* 6.0% in FY2025
* 54 bps margin improvement 
* 37% increase in EBITDA 
* EBITDA at H474 crores in FY2025

📄 Source pages: [70, 72, 226]
------------------------------------------------------------


'* 5.4% in FY2024\n* 6.0% in FY2025\n* 54 bps margin improvement \n* 37% increase in EBITDA \n* EBITDA at H474 crores in FY2025'

In [23]:
ask("How many new brands launched in FY2025?")

Q: How many new brands launched in FY2025?

A:
* 800 new fashion brands 
* 8,600+ total brands 
* Over 4,400 global and domestic brands 
* 250 globally renowned brands in GCC 
* Highest-ever brands launches in a financial year

📄 Source pages: [1, 10, 18, 66]
------------------------------------------------------------


'* 800 new fashion brands \n* 8,600+ total brands \n* Over 4,400 global and domestic brands \n* 250 globally renowned brands in GCC \n* Highest-ever brands launches in a financial year'

In [17]:
ask("What are the foreign brands that are listed in Nykaa?")

Q: What are the foreign brands that are listed in Nykaa?

A:
* Chanel 
* 0.42% Nykaa International UK Limited 
* 1.89% Nykaa International UK Limited 
* 2.41% Nykaa International UK Limited 
* 6.26% Nykaa International UK Limited

📄 Source pages: [12, 67, 229, 313]
------------------------------------------------------------


'* Chanel \n* 0.42% Nykaa International UK Limited \n* 1.89% Nykaa International UK Limited \n* 2.41% Nykaa International UK Limited \n* 6.26% Nykaa International UK Limited'

In [14]:
ask("Which is the most profitable brand listed in Nykaa?")

Q: Which is the most profitable brand listed in Nykaa?

A:
* 363.16%: Nykaa E-retail Limited 
* 71.90%: Iluminar Media Limited 
* 21.37%: FSN Brands Marketing Private Limited 
* 142.03%: FSN E-Commerce Ventures Limited 
* 20.67%: FSN Brands Marketing Private Limited

📄 Source pages: [5, 8, 313]
------------------------------------------------------------


'* 363.16%: Nykaa E-retail Limited \n* 71.90%: Iluminar Media Limited \n* 21.37%: FSN Brands Marketing Private Limited \n* 142.03%: FSN E-Commerce Ventures Limited \n* 20.67%: FSN Brands Marketing Private Limited'

In [15]:
ask("How many total brands are listed in Nykaa?")

Q: How many total brands are listed in Nykaa?

A:
* 4,400 brands
* 12 owned brands 
* Numerous premium international brands 
* House of Nykaa brands 
* Including Chanel

📄 Source pages: [9, 12, 65, 67]
------------------------------------------------------------


'* 4,400 brands\n* 12 owned brands \n* Numerous premium international brands \n* House of Nykaa brands \n* Including Chanel'

In [16]:
ask("How many total international brands are listed in Nykaa?")

Q: How many total international brands are listed in Nykaa?

A:
* 1,000+ 
* 500+ 
* 4,400+ total brand partners
* 100+ House of Nykaa – fashion brands
* 5 House of Nykaa brands are not international, exact total international is not available

📄 Source pages: [6, 12, 67, 313]
------------------------------------------------------------


'* 1,000+ \n* 500+ \n* 4,400+ total brand partners\n* 100+ House of Nykaa – fashion brands\n* 5 House of Nykaa brands are not international, exact total international is not available'

In [23]:
ask("Which are the most profitable brands listed in Nykaa?")

Q: Which are the most profitable brands listed in Nykaa?

A:
* 363.16%: Nykaa E-retail Limited 
* 71.90%: Iluminar Media Limited 
* 21.37%: FSN Brands Marketing Private Limited 
* 142.03%: FSN E-Commerce Ventures Limited 
* 134.86%: FSN E-Commerce Ventures Limited

📄 Source pages: [5, 65, 313]
------------------------------------------------------------


'* 363.16%: Nykaa E-retail Limited \n* 71.90%: Iluminar Media Limited \n* 21.37%: FSN Brands Marketing Private Limited \n* 142.03%: FSN E-Commerce Ventures Limited \n* 134.86%: FSN E-Commerce Ventures Limited'

In [28]:
ask("What is the total Revenue of Nykaa in FY2024-2025?")

Q: What is the total Revenue of Nykaa in FY2024-2025?

A:
* $935 million 
* 7,950 crores 
* 15,604 crores GMV 
* 65 crores others’ operating revenue 
* 4,473 crores cost of goods sold

📄 Source pages: [10, 11, 70, 226]
------------------------------------------------------------


'* $935 million \n* 7,950 crores \n* 15,604 crores GMV \n* 65 crores others’ operating revenue \n* 4,473 crores cost of goods sold'

In [26]:
ask("What percentage of revenue is from online and what percentage is from ofline?")

Q: What percentage of revenue is from online and what percentage is from ofline?

A:
* 94% from Retail sale via e-commerce (online)
* 6% from Wholesale of cosmetics (Offline - Own brands) 
* 30% of Indian BPC market size is offline 
* 18% of Indian BPC market size is online 
* 100 is mentioned but not related to percentage of revenue

📄 Source pages: [63, 130, 217, 280]
------------------------------------------------------------


'* 94% from Retail sale via e-commerce (online)\n* 6% from Wholesale of cosmetics (Offline - Own brands) \n* 30% of Indian BPC market size is offline \n* 18% of Indian BPC market size is online \n* 100 is mentioned but not related to percentage of revenue'

In [29]:
ask("Who is the CEO of Nykaa?")

Q: Who is the CEO of Nykaa?

A:
* Falguni Nayar, Executive Chairperson, Managing Director & CEO
* Adwaita Nayar, Executive Director - FSN E-Commerce Ventures Limited, Managing Director & Chief Executive Officer - Nykaa Fashion
* Anchit Nayar, Executive Director - FSN E-Commerce Ventures Limited, Managing Director & Chief Executive Officer – Nykaa E-Retail 
* Vishal Gupta, Chief Executive Officer - Nykaa Distribution

📄 Source pages: [8, 87, 103, 326]
------------------------------------------------------------


'* Falguni Nayar, Executive Chairperson, Managing Director & CEO\n* Adwaita Nayar, Executive Director - FSN E-Commerce Ventures Limited, Managing Director & Chief Executive Officer - Nykaa Fashion\n* Anchit Nayar, Executive Director - FSN E-Commerce Ventures Limited, Managing Director & Chief Executive Officer – Nykaa E-Retail \n* Vishal Gupta, Chief Executive Officer - Nykaa Distribution'

In [31]:
ask("What is the growth strategy of Nykaa?")



Q: What is the growth strategy of Nykaa?

A:
24% growth over the previous year
6% EBITDA margin 
37% increase in EBITDA 
Driving sustainable growth 
Investing in future-ready capabilities

📄 Source pages: [8, 9, 48, 151]
------------------------------------------------------------


'24% growth over the previous year\n6% EBITDA margin \n37% increase in EBITDA \nDriving sustainable growth \nInvesting in future-ready capabilities'

In [32]:
ask("How many employees are working with Nykaa?")

Q: How many employees are working with Nykaa?

A:
* 3,735 Total employees
* 1,322 New employees onboarded 
* 3,094 Number of employees trained 
* 43% Female workforce 
* H666 Cr Spend on employees

📄 Source pages: [35, 43, 102, 148]
------------------------------------------------------------


'* 3,735 Total employees\n* 1,322 New employees onboarded \n* 3,094 Number of employees trained \n* 43% Female workforce \n* H666 Cr Spend on employees'

In [30]:
ask("What is the loss percentage of Nykaa in FY2024-FY2025?")


Q: What is the loss percentage of Nykaa in FY2024-FY2025?

A:
* 9.21% loss for Nykaa Fashion Limited in FY2025
* 22.02% for Parent: FSN E-Commerce Ventures Limited 
* 45.90% for Nykaa E-retail Limited 
* 19.01% for Nykaa Fashion Limited 
* 9.74% for FSN Brands Marketing Private Limited

📄 Source pages: [70, 158, 226, 313]
------------------------------------------------------------


'* 9.21% loss for Nykaa Fashion Limited in FY2025\n* 22.02% for Parent: FSN E-Commerce Ventures Limited \n* 45.90% for Nykaa E-retail Limited \n* 19.01% for Nykaa Fashion Limited \n* 9.74% for FSN Brands Marketing Private Limited'

In [31]:
ask("How much money Nykaa burnt in marketing and brand promotion in FY2024-2025?")


Q: How much money Nykaa burnt in marketing and brand promotion in FY2024-2025?

A:
* 21.29 crores: Banner advertisement expenses
* 111.80 crores: (Recovery) / reimbursement of expenses 
* 5.67 crores: Commission - Financial Guarantee 
* 38.53 crores: Discount expenses 
* 107.85 crores: Sales (net of discounts)

📄 Source pages: [9, 70, 124, 226]
------------------------------------------------------------


'* 21.29 crores: Banner advertisement expenses\n* 111.80 crores: (Recovery) / reimbursement of expenses \n* 5.67 crores: Commission - Financial Guarantee \n* 38.53 crores: Discount expenses \n* 107.85 crores: Sales (net of discounts)'

In [33]:
ask("Who is the brand ambassador of Nykaa?")


Q: Who is the brand ambassador of Nykaa?

A:
* Rasha Thadani 
* She embodies the young, fun spirit that the brand aims to capture.

📄 Source pages: [5, 14, 67, 326]
------------------------------------------------------------


'* Rasha Thadani \n* She embodies the young, fun spirit that the brand aims to capture.'